# Network Security Traffic Analyzer — Public Demo
Synthetic-only demonstration. No private course materials are included.


### Setup
Colab downloads the pinned public baseline automatically. Local Jupyter uses the extracted project.

In [ ]:
from pathlib import Path
import sys, subprocess, tempfile
from importlib.util import find_spec

BASELINE = "28829fc98de74a22a70ac8371256f460d5788bf6"
REPO = "https://github.com/GitMasterMindX/Network_Security_Traffic_Analyzer.git"
if find_spec("google.colab") is not None:
    root = Path(tempfile.mkdtemp(prefix="network_security_")) / "repo"
    subprocess.run(["git", "clone", "--quiet", "--no-checkout", REPO, str(root)], check=True)
    subprocess.run(["git", "-C", str(root), "checkout", "--quiet", "--detach", BASELINE], check=True)
    actual = subprocess.check_output(["git", "-C", str(root), "rev-parse", "HEAD"], text=True).strip()
    if actual != BASELINE:
        raise RuntimeError("Baseline verification failed")
else:
    root = Path.cwd()
    if not (root / "analyzer_v04.py").is_file():
        root = root / "Network_Security_Traffic_Analyzer_Public_Release"
for required in ("analyzer_v04.py", "synthetic_data", "tests"):
    if not (root / required).exists():
        raise FileNotFoundError(f"Missing {required}; launch from the extracted project folder")
sys.path.insert(0, str(root))
print("Project:", root)


In [ ]:
from analyzer_v04 import extract_tcpdump, extract_wireshark, summarize, save
sources={'dns_icmp':('dns_icmp.txt',extract_tcpdump),'http':('http.txt',extract_tcpdump),'syn':('syn.txt',extract_wireshark)}
evidence={}
for case,(filename,parser) in sources.items():
    events,rejected=parser((root/'synthetic_data'/filename).read_text(encoding='utf-8'),filename)
    evidence[case]={'events':events,'rejected':rejected}
    print(case,'parsed:',len(events),'rejected:',len(rejected))


In [ ]:
report=summarize(evidence)
report['evidence_origin']='synthetic_demo'
report['limitations']=['Synthetic demonstration, not evidence of a real incident.']
for case,item in report['reports'].items():
    print(case,item['counts'],[(f['id'],f['classification']) for f in item['findings']])
output=root/'reports'/'synthetic_demo_report.json'
output.parent.mkdir(exist_ok=True)
save(report,output)
print('Saved:',output)


### Automated tests
Install pytest if missing, then run the public regression suite.

In [ ]:
import importlib.util
if importlib.util.find_spec("pytest") is None:
    subprocess.run([sys.executable, "-m", "pip", "install", "pytest==9.2.0"], check=True)
subprocess.run([sys.executable, "-m", "pytest", "-q"], cwd=root, check=True)
print("Regression tests: PASS")
